In [0]:
catalog = dbutils.widgets.get("catalog");
spark.sql(f"USE CATALOG {catalog}");
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

Importation Bibliothèque

In [0]:
from pyspark.sql.functions import *

Phase transformation

In [0]:
df_depart_raws = spark.read.table("bronze.depart_raws")
display(df_depart_raws.limit(5))

In [0]:
df_depart_raws.printSchema() # Correction de la colonned date

In [0]:
df_depart_raws.describe().show() # Absence de valeur manquante

In [0]:
# Verification des doublons

#df_depart_raws.groupBy(df_depart_raws.columns).count().filter(col("count") > 1).count() # Suppression des doublons

In [0]:
# Verificaation du nombre caractère
#df_depart_raws.filter(length(col("origin")) != 3).show()

#df_depart_raws.filter(length(col("destination")) != 3).show()

In [0]:
# Standardisation
#df_depart_stand = df_depart_raws.withColumn("origin",upper(trim(col("origin")))).withColumn("destination",upper(trim(col("destination"))))

In [0]:
#df_depart_raws.filter(col("origin") == col("destination")).show() #  Aucune ressemblance

Qualité des données

Table validé

In [0]:
df_depart_transf = (
    df_depart_raws
    .dropna()
    .withColumn("origin",upper(trim(col("origin"))))
    .withColumn("destination",upper(trim(col("destination"))))
    .filter(length(col("origin")) == 3)
    .filter(length(col("destination")) == 3)
    .filter(col("origin") != col("destination"))
    .dropDuplicates()
)

#df_depart_transf.show(5)

In [0]:
df_depart_rejet= (
    df_depart_raws
    .withColumn(
        "motif_rejet",
        when(col("origin").isNull() | col("destination").isNull(),"Valeur nulle")
        .when(length(col("origin")) != 3,"taille caractère incorrecte")
        .when(length(col("destination")) != 3,"taille caractère incorrecte")
        .when(col("origin") == col("destination"),"Même origine et destination")
    )
    .filter(col("motif_rejet").isNotNull())


)

In [0]:
df_doublons = (
    df_depart_raws
    .groupBy(df_depart_raws.columns)
    .count()
    .filter(col("count") > 1)
    .drop("count")
    .withColumn("motif_rejet",lit("En Doublon"))
)


In [0]:
#display(df_doublons.limit(5))

Tabble des rejets

In [0]:
df_depart_rejet = df_depart_rejet.unionByName(df_doublons)

#display(df_depart_rejet.limit(5))

In [0]:
# Sauvegarde

df_depart_rejet.write.format("delta").mode("overwrite").saveAsTable("silver.silver_depart_rejet")

In [0]:
 data_depart_valide =(
    df_depart_transf
    .withColumn("date_str",lpad(col("date").cast("string"),8,"0"))
    .withColumn("localdate", to_timestamp(concat(
        lit("2014-"),
        substring(col("date_str"),1,2),
        lit("-"),
        substring(col("date_str"),3,2),
        lit(" "),
        substring(col("date_str"),5,2),
        lit(":"),
        substring(col("date_str"),7,2)
        )))
    .drop("date","date_str")
)

Lecture de la table Airport


In [0]:
airport_raws = spark.read.table("bronze.airport_raws")

#airport_raws.show(5)

Jointure

In [0]:
flights_silver = (
data_depart_valide.alias("dpt")
.join(airport_raws.alias("air_src"),col("dpt.origin") == col("air_src.IATA"),"left")
.join(airport_raws.alias("air_dest"),col("dpt.destination") == col("air_dest.IATA"),"left")
.select(
    col("dpt.localdate"),
    col("dpt.delay"),
    col("dpt.distance"),
    col("dpt.origin").alias("origin"),
    col("air_src.City").alias("origin_town"),
    col("dpt.destination").alias("destination"),
    col("air_dest.City").alias("destination_town")
)
.withColumn('numberMonth', month(col('localdate')))
.withColumn('month_name',date_format(col('localdate'), 'MMMM'))
)

In [0]:
flights_silver.write.format("delta").mode("overwrite").saveAsTable("silver.silver_city_flights")